In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
"""
FDIC Banking Analysis: How Interest Rates Impact Top US Bank Revenue (2015–2025)
Author: Giovanni Del Angel
Description: Loads, cleans, and transforms FDIC quarterly call report data for
             the top 10 US banks, joined with Federal Funds Rate data.
"""

import os
import glob
import pandas as pd
import duckdb

# ── 1. LOAD DATA ──────────────────────────────────────────────────────────────

FOLDER = '/content/drive/MyDrive/Co-Op/FDIC_Banking_Data'

all_files = glob.glob(os.path.join(FOLDER, '*.csv'))
print(f"Found {len(all_files)} files")

df = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in all_files],
    ignore_index=True
)
print(f"Total rows: {len(df):,} | Total columns: {len(df.columns)}")


# ── 2. FILTER TO TOP 10 BANKS ─────────────────────────────────────────────────
# Note: FDIC uses abbreviated naming conventions, not official legal names

banks = [
    'JPMORGAN CHASE BANK NA',
    'BANK OF AMERICA NA',
    'WELLS FARGO BANK NA',
    'CITIBANK NATIONAL ASSN',
    'GOLDMAN SACHS BANK USA',
    'MORGAN STANLEY BANK NA',
    'PNC BANK NATIONAL ASSN',
    'TRUIST BANK',
    'CAPITAL ONE NATIONAL ASSN',
    'CAPITAL ONE BANK USA NA'
]

df_top10 = df[df['NAME'].isin(banks)].copy()
print(f"\nRows after filter: {len(df_top10)}")
print(df_top10['NAME'].value_counts())


# ── 3. SELECT KEY COLUMNS ─────────────────────────────────────────────────────

cols = {
    'NAME':    'Bank name',
    'REPDTE':  'Report date (quarter)',
    'ASSET':   'Total assets',
    'NETINC':  'Net income (cumulative YTD)',
    'INTINC':  'Interest income (cumulative YTD)',
    'EINTEXP': 'Interest expense (cumulative YTD)',
    'NIM':     'Net interest margin',
    'ROA':     'Return on assets (%)',
    'ROE':     'Return on equity (%)',
    'DEP':     'Total deposits',
    'LNLSNET': 'Net loans and leases',
    'EQ':      'Total equity'
}

df_clean = df_top10[list(cols.keys())].copy()
df_clean['REPDTE'] = pd.to_datetime(df_clean['REPDTE'], format='%Y%m%d')
df_clean = df_clean.sort_values(['NAME', 'REPDTE']).reset_index(drop=True)


# ── 4. ADD FEDERAL FUNDS RATE ─────────────────────────────────────────────────

fed_rates = {
    '2015-03-31': 0.11, '2015-06-30': 0.13, '2015-09-30': 0.14, '2015-12-31': 0.24,
    '2016-03-31': 0.36, '2016-06-30': 0.38, '2016-09-30': 0.40, '2016-12-31': 0.54,
    '2017-03-31': 0.79, '2017-06-30': 1.04, '2017-09-30': 1.15, '2017-12-31': 1.30,
    '2018-03-31': 1.51, '2018-06-30': 1.82, '2018-09-30': 2.02, '2018-12-31': 2.27,
    '2019-03-31': 2.41, '2019-06-30': 2.38, '2019-09-30': 2.04, '2019-12-31': 1.55,
    '2020-03-31': 0.65, '2020-06-30': 0.08, '2020-09-30': 0.09, '2020-12-31': 0.09,
    '2021-03-31': 0.07, '2021-06-30': 0.08, '2021-09-30': 0.08, '2021-12-31': 0.08,
    '2022-03-31': 0.20, '2022-06-30': 1.21, '2022-09-30': 2.56, '2022-12-31': 3.78,
    '2023-03-31': 4.65, '2023-06-30': 5.08, '2023-09-30': 5.33, '2023-12-31': 5.33,
    '2024-03-31': 5.33, '2024-06-30': 5.33, '2024-09-30': 4.96, '2024-12-31': 4.48,
    '2025-03-31': 4.33, '2025-06-30': 4.33, '2025-09-30': 4.08, '2025-12-31': 3.72
}

fed_df = pd.DataFrame(list(fed_rates.items()), columns=['REPDTE', 'FED_RATE'])
fed_df['REPDTE'] = pd.to_datetime(fed_df['REPDTE'])

df_merged = df_clean.merge(fed_df, on='REPDTE', how='left')

nulls = df_merged['FED_RATE'].isna().sum()
print(f"\nMissing Fed Rate values: {nulls}")


# ── 5. CLEAN & TRANSFORM WITH DUCKDB SQL ──────────────────────────────────────

con = duckdb.connect()
con.register('banks_raw', df_merged)

df_final = con.execute("""
    SELECT
        CASE NAME
            WHEN 'JPMORGAN CHASE BANK NA'    THEN 'JPMorgan Chase'
            WHEN 'BANK OF AMERICA NA'         THEN 'Bank of America'
            WHEN 'WELLS FARGO BANK NA'        THEN 'Wells Fargo'
            WHEN 'CITIBANK NATIONAL ASSN'     THEN 'Citibank'
            WHEN 'GOLDMAN SACHS BANK USA'     THEN 'Goldman Sachs'
            WHEN 'MORGAN STANLEY BANK NA'     THEN 'Morgan Stanley'
            WHEN 'PNC BANK NATIONAL ASSN'     THEN 'PNC Bank'
            WHEN 'TRUIST BANK'                THEN 'Truist Bank'
            WHEN 'CAPITAL ONE NATIONAL ASSN'  THEN 'Capital One'
            WHEN 'CAPITAL ONE BANK USA NA'    THEN 'Capital One USA'
        END                                        AS bank_name,

        CAST(REPDTE AS DATE)                       AS report_date,
        YEAR(CAST(REPDTE AS DATE))                 AS year,
        QUARTER(CAST(REPDTE AS DATE))              AS quarter,

        ROUND(ASSET    / 1000000.0, 2)             AS total_assets_billions,
        ROUND(NETINC   / 1000000.0, 2)             AS net_income_billions,
        ROUND(INTINC   / 1000000.0, 2)             AS interest_income_billions,
        ROUND(EINTEXP  / 1000000.0, 2)             AS interest_expense_billions,
        ROUND(DEP      / 1000000.0, 2)             AS total_deposits_billions,
        ROUND(LNLSNET  / 1000000.0, 2)             AS net_loans_billions,
        ROUND(EQ       / 1000000.0, 2)             AS total_equity_billions,
        ROUND(NIM      / 1000000.0, 2)             AS net_interest_margin_billions,
        ROUND(ROA, 2)                               AS return_on_assets_pct,
        ROUND(ROE, 2)                               AS return_on_equity_pct,
        FED_RATE                                    AS fed_funds_rate_pct

    FROM banks_raw
    WHERE NAME IS NOT NULL
    ORDER BY bank_name, report_date
""").df()

print(f"\nFinal dataset: {len(df_final):,} rows x {len(df_final.columns)} columns")
print(df_final.head())


# ── 6. SAVE OUTPUT ────────────────────────────────────────────────────────────

OUTPUT_PATH = os.path.join(FOLDER, 'bank_data_final.csv')
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to: {OUTPUT_PATH}")

Found 44 files
Total rows: 231,650 | Total columns: 161

Rows after filter: 408
NAME
JPMORGAN CHASE BANK NA       44
BANK OF AMERICA NA           44
WELLS FARGO BANK NA          44
CAPITAL ONE NATIONAL ASSN    44
PNC BANK NATIONAL ASSN       44
CITIBANK NATIONAL ASSN       44
MORGAN STANLEY BANK NA       44
GOLDMAN SACHS BANK USA       44
CAPITAL ONE BANK USA NA      31
TRUIST BANK                  25
Name: count, dtype: int64

Missing Fed Rate values: 0

Final dataset: 408 rows x 15 columns
         bank_name report_date  year  quarter  total_assets_billions  \
0  Bank of America  2015-03-31  2015        1                1599.75   
1  Bank of America  2015-06-30  2015        2                1606.23   
2  Bank of America  2015-09-30  2015        3                1616.43   
3  Bank of America  2015-12-31  2015        4                1639.31   
4  Bank of America  2016-03-31  2016        1                1653.95   

   net_income_billions  interest_income_billions  interest_expense_bil